# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Manjit-yadav/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### 1. Unit of analysis + time window

**Unit of analysis:** One row in `fact_content_daily_performance` represents one content item for one client on one report date.

For my Search Intelligence lane, I treat the **content item** as the unit I want to prioritize for review or refresh.

**Time window:** I use **February 2026 as the feature window** and **March 2026 as the future outcome/label window**. Only information available through the February decision cutoff is used as a feature. March information is reserved for the outcome.

This separation is important because a feature should represent information that would have been available when the content team made the prioritization decision.


In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)
print("Token starts with:", HF_TOKEN[:5] if HF_TOKEN else None)

In [ ]:
import os
import getpass
import duckdb
import pandas as pd

def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    return getpass.getpass("Enter your Hugging Face READ token: ")

HF_TOKEN = get_hf_token()

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"
DIM_CONTENT = f"{REL}/dim_content.parquet"
DIM_CLIENTS = f"{REL}/dim_clients.parquet"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Connected to FlyRank warehouse")
print("Feature window: February 2026")
print("Label window: March 2026")

### 2. Fields: feature / label / context / excluded

**Features**

I will use information available before the decision cutoff, including:

* `impressions`
* `clicks`
* `avg_position`
* `content_age_days`
* `word_count`

These describe the page's observed search performance and content characteristics before the future outcome is measured.

**Label**

My label is whether the content meets the defined future decline criterion in the March 2026 outcome window.

**Context**

I will keep the following as context for grouping, joining, or interpreting results:

* `client_id`
* `content_id`
* `report_date`
* content metadata from `dim_content`

These identifiers help connect and describe observations but are not intended to be predictive features.

**Excluded**

I deliberately exclude `trend_direction` and `trend_pct` from the feature set because they are derived from performance changes and can directly reveal the future outcome. Including them would create label leakage.

I also exclude future March performance measures from the feature set because they would not have been known at the February decision cutoff.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
DESCRIBE SELECT *
FROM {FEB}
LIMIT 1
""").df()

### Query 1 — Verify the grain

I claim that the daily performance table has one row per report date, client, and content item. I therefore check whether `(report_date, client_id, content_id)` has duplicates in the February feature window.

A zero-row result means no duplicate combinations were found in this checked window.


In [ ]:
grain_check = con.sql(f"""
SELECT
    report_date,
    client_id,
    content_id,
    COUNT(*) AS row_count
FROM {FEB}
GROUP BY
    report_date,
    client_id,
    content_id
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 20
""").df()

grain_check

### Query 2 — Row count and date span

I check the number of rows and the first and last report dates in the February feature window. This verifies that the selected month actually covers the intended decision period.


In [ ]:
window_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {FEB}
""").df()

window_check

### Query 3 — Availability

I use `IS TRUE` rather than `= TRUE` so that the check handles SQL `NULL` values explicitly. The output shows how many rows have the required data available in the feature window and how many are unavailable or missing.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
DESCRIBE SELECT *
FROM {FEB}
LIMIT 1
""").df()

availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS NOT TRUE
    ) AS unavailable_or_missing_rows
FROM {FEB}
""").df()

availability_check

### Five-feature frame

I construct a small feature frame using only information available in the February feature window. The features are aggregated to the content-item level so that each content item is represented once for the prioritization task.

The five features are:

1. **Impressions** — measures observed search visibility before the decision.
2. **Clicks** — measures observed search traffic before the decision.
3. **Average position** — describes the page's observed search position before the decision.
4. **Content age** — describes how long the content had existed at the decision cutoff.
5. **Word count** — describes the size of the content available at the decision cutoff.

Each feature is available at the decision moment because it is calculated only from information known before the March outcome window.


In [ ]:
feature_frame = con.sql(f"""
WITH feb AS (
    SELECT
        client_id,
        content_id,
        SUM(impressions) AS impressions,
        SUM(clicks) AS clicks,
        AVG(avg_position) AS avg_position
    FROM {FEB}
    GROUP BY
        client_id,
        content_id
)

SELECT
    f.client_id,
    f.content_id,
    f.impressions,
    f.clicks,
    f.avg_position,
    c.content_age_days,
    c.word_count
FROM feb f
LEFT JOIN read_parquet('{DIM_CONTENT}') c
    ON f.content_id = c.content_id
""").df()

feature_frame.head()

In [ ]:
print("Feature frame shape:", feature_frame.shape)
print("Feature columns:")
print(feature_frame.columns.tolist())

In [ ]:
feature_availability = pd.DataFrame({
    "feature": [
        "impressions",
        "clicks",
        "avg_position",
        "content_age_days",
        "word_count"
    ],
    "available_when": [
        "Known by the February decision cutoff because it is calculated from earlier search observations.",
        "Known by the February decision cutoff because the clicks occurred before the decision.",
        "Known by the February decision cutoff because it summarizes earlier observed search position.",
        "Known by the February decision cutoff because the content's age is already known.",
        "Known by the February decision cutoff because the content length is already known."
    ]
})

feature_availability

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score

# Example only — use your actual label column
X = feature_frame[
    [
        "impressions",
        "clicks",
        "avg_position",
        "content_age_days",
        "word_count"
    ]
]

y = feature_frame["label"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

model.fit(X_train, y_train)

honest_pred = model.predict(X_test)

honest_precision = precision_score(
    y_test,
    honest_pred,
    zero_division=0
)

print("Honest precision:", honest_precision)

In [ ]:
label = ...

In [ ]:
X_leaky = X.copy()

X_leaky["leak"] = (
    march_outcome["trend_direction"] == "down"
).astype(int)

In [ ]:
model_leaky = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

model_leaky.fit(X_leaky_train, y_train)

leaky_pred = model_leaky.predict(X_leaky_test)

leaky_precision = precision_score(
    y_test,
    leaky_pred,
    zero_division=0
)

print("Leaky precision:", leaky_precision)

In [ ]:
leakage_results = pd.DataFrame({
    "experiment": [
        "Honest features",
        "Features + label-derived leak"
    ],
    "score": [
        honest_precision,
        leaky_precision
    ]
})

leakage_results

### 4. Data limits

**Unbalanced history:** The warehouse is an unbalanced panel, so different clients have different amounts of historical data. This means the same calendar window may not represent the same amount of historical context for every client.

**Data availability:** Some early observations may have only certain data sources available. Therefore, missing or unavailable GSC/GA4 data can affect which content items can be included in a feature set.

**Window overlap:** Feature and outcome windows must remain separate. If information from the future outcome window is accidentally included in the features, the model can leak the answer and produce an overly optimistic score.

Therefore, this analysis provides **observed and directional decision-support evidence**, not causal proof of why a page changed or a prediction of what Google will do.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.